In [12]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [13]:
## Load the trained model, scaler pickle and onehot encoder pickle files
model = load_model('model.h5')    ## load_model() method is used to load the trained model. It takes the name of the file as an argument. It returns a model object.

## load the encoders and scalers
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)    ## pickle.load() method is used to load the pickle file. It takes the file object as an argument. It returns the object stored in the pickle file.

with open('laber_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)    ## pickle.load() method is used to load the pickle file. It takes the file object as an argument. It returns the object stored in the pickle file.

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)    ## pickle.load() method is used to load the pickle file. It takes the file object as an argument. It returns the object stored in the pickle file.

In [20]:
# Check the label encoder classes
print("Label Encoder Classes:", label_encoder_gender.classes_)
print("Classes dtype:", label_encoder_gender.classes_.dtype)

Label Encoder Classes: [0 1]
Classes dtype: int64


In [18]:
# Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [21]:
# Create DataFrame from input data
input_df = pd.DataFrame([input_data])

# Encode Gender manually (0 for Female, 1 for Male)
gender_mapping = {'Female': 0, 'Male': 1}
input_df['Gender'] = input_df['Gender'].map(gender_mapping)

# One hot encode the 'Geography' column
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

# Combine the encoded columns
input_df = pd.concat([input_df.drop(['Geography'], axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [22]:
# Scale the features
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [ ]:
# Make prediction
prediction = model.predict(input_scaled)
print(prediction)
prediction_proba = prediction[0][0]
print(prediction_proba)

print(f"Churn Probability: {prediction_proba:.4f}")
print(f"Churn Prediction: {'Yes' if prediction_proba > 0.5 else 'No'}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
[[0.02418415]]
0.024184149
Churn Probability: 0.0242
Churn Prediction: No


: 